# Mixed-language status example

This notebook ports `workflow-mount.py` from file mounts to Jupyter widgets. A Python transformer makes a random text from a `length` and a `seed`; a Bash transformer then edits that text. Each step sleeps for 5 seconds.

Integer cells become sliders and text cells become text fields. The two transformer codes can be edited in text areas, and the two results are read-only text fields. No files are mounted.

Run the cells with the **seamless1** Python kernel. The widgets need the optional `seamless-workflow[jupyter]` dependencies, and viewing and moving them needs JupyterLab or another widget frontend.

## 1. The Python transformer

The function is stored in a `python` cell, so that it can be edited. `set_buffer` copies the transformer's code into the cell, and the cell is then connected as the transformer's code. The code is available at once; no `compute()` is needed. `length` and `seed` start at the values of the mount version.

In [1]:
from seamless.workflow import Context, Cell, Transformer

ctx = Context()

def random_text(length:int, seed:int):
    import time
    import random
    time.sleep(5)
    random.seed(seed)
    alphabet = [chr(n) for n in range(ord('A'), ord('Z')+1)]
    letters = random.choices(alphabet, k=length)
    return "".join(letters)

ctx.random_text = random_text
ctx.random_text_code = Cell("python")
ctx.random_text_code.set_buffer(ctx.random_text.code)
ctx.random_text.code = ctx.random_text_code
ctx.length = Cell("int")
ctx.length.set(12)
ctx.seed = Cell("int")
ctx.seed.set(128)
ctx.random_text.pins.length = ctx.length
ctx.random_text.pins.seed = ctx.seed

ctx.text1 = Cell("text")
ctx.text1 = ctx.random_text.result

## 2. The Bash transformer

`modify_text` replaces every `C` by `Y` in `text1`. Its `text1` pin is declared before its code is assigned: a Bash transformer that has code and no pins is already runnable, and would start a throw-away run that fails because there is no `text1` yet.

In [2]:
bash_code = """
sleep 5
sed 's/C/Y/g' text1 > RESULT
"""
ctx.modify_text_code = Cell("text")
ctx.modify_text_code.set(bash_code)

ctx.modify_text = Transformer("bash")
ctx.modify_text.pins.text1 = ctx.text1
ctx.modify_text.code = ctx.modify_text_code

ctx.text2 = Cell("text")
ctx.text2 = ctx.modify_text.result

## 3. Link the cells to widgets

`traitlet(cell)` returns the cell's widget hub. `link(widget)` is bidirectional: the cell's value is shown in the widget, and a change of the widget is written to the cell. `connect(widget)` is one-way, from the cell to the widget; it is the only form that is legal on `text1` and `text2`, which are computed. Their text fields are `disabled`, so they cannot be edited.

Each accepted input starts a run of 5 seconds. The sliders and the text areas therefore use `continuous_update=False`: a value is sent when a slider is released, or when a text area loses focus (click outside it). A widget only shows values that are complete; while a result is being recomputed, or after it has failed, it keeps its last value.

In [3]:
import ipywidgets as widgets
from IPython.display import display
from seamless_workflow.jupyter import traitlet

style = {"description_width": "10em"}
wide = widgets.Layout(width="45em")

length = widgets.IntSlider(min=1, max=40, description="length", style=style, layout=wide, continuous_update=False)
seed = widgets.IntSlider(min=0, max=1000, description="seed", style=style, layout=wide, continuous_update=False)
random_text_code = widgets.Textarea(description="random_text_code", rows=10, style=style, layout=wide, continuous_update=False)
text1 = widgets.Text(description="text1", disabled=True, style=style, layout=wide)
modify_text_code = widgets.Textarea(description="modify_text_code", rows=4, style=style, layout=wide, continuous_update=False)
text2 = widgets.Text(description="text2", disabled=True, style=style, layout=wide)

traitlet(ctx.length).link(length)
traitlet(ctx.seed).link(seed)
traitlet(ctx.random_text_code).link(random_text_code)
traitlet(ctx.text1).connect(text1)
traitlet(ctx.modify_text_code).link(modify_text_code)
traitlet(ctx.text2).connect(text2)

display(length, seed, random_text_code, text1, modify_text_code, text2)

IntSlider(value=12, continuous_update=False, description='length', layout=Layout(width='45em'), max=40, min=1,…

IntSlider(value=128, continuous_update=False, description='seed', layout=Layout(width='45em'), max=1000, style…

Textarea(value='def random_text(length:int, seed:int):\n    import time\n    import random\n    time.sleep(5)\…

Text(value='YGNDAEQGRSOB', description='text1', disabled=True, layout=Layout(width='45em'), style=TextStyle(de…

Textarea(value="\nsleep 5\nsed 's/C/Y/g' text1 > RESULT", continuous_update=False, description='modify_text_co…

Text(value='', description='text2', disabled=True, layout=Layout(width='45em'), style=TextStyle(description_wi…

## 4. Use it

Move a slider or edit a code, and watch the fields. A change of `length` or `seed` updates `text1` after 5 seconds and `text2` after 10 seconds; a change of `modify_text_code` updates `text2` after 5 seconds. No cell is running meanwhile, so the widgets stay responsive.

The first cell below is optional: it waits, without blocking the kernel, until the graph is quiet and the results have reached the widgets. (`ctx.computation()` alone would not wait for a widget change that was made a moment ago, nor for the delivery.) The second reads the node states at any time. If a code no longer works, its transformer fails, and the outputs below it keep their last value; the node states then show the error.

In [4]:
_ = await ctx.mounts.synchronization(timeout=60)

In [6]:
for name in ("length", "seed", "random_text_code", "random_text", "text1",
             "modify_text_code", "modify_text", "text2"):
    node = getattr(ctx, name)
    line = f"{name:18} {node.state}"
    if node.exception:
        line += "  " + str(node.exception).strip().splitlines()[-1]
    print(line)

length             complete
seed               complete
random_text_code   complete
random_text        computing
text1              waiting
modify_text_code   complete
modify_text        waiting
text2              waiting


## 5. Clean up

Leave the following cell unexecuted until you have finished with the widgets. Closing the Context detaches them from their cells; they keep the values they last showed.

In [ ]:
ctx.close()
print("Context closed")